# Federated Learning with Hard Sample Removal

## Strategy Overview
This notebook implements intelligent hard sample detection and removal for federated learning with limited data:

### Detection Strategy:
1. **Profiling Phase (First 15 rounds)**: Train normally and collect per-sample statistics
2. **Sample Analysis**: Track for each sample:
   - Average loss across profiling rounds
   - Average prediction confidence
   - Prediction consistency (variance of predicted class)
3. **Conservative Removal**: Remove only the worst 10-15% of samples based on multiple criteria
4. **Continued Training**: Train with cleaned data for remaining rounds

### Hard Sample Criteria:
- **High Loss**: Samples with consistently high loss (top 20% in loss)
- **Low Confidence**: Model never confident about prediction (avg confidence < 40%)
- **Inconsistent Predictions**: Predictions change frequently across rounds (variance > threshold)
- **Combined Score**: Weight all three criteria to identify truly problematic samples

### Configuration:
- 100 clients, each with 100 samples (10 per class)
- 15 rounds profiling, then remove samples, then 85 rounds training
- Remove worst 10-15% samples (10-15 samples per client)
- Conservative approach to preserve data

In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from collections import defaultdict
import os
from tqdm import tqdm

# Suppress TensorFlow warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
tf.get_logger().setLevel('ERROR')

print("TensorFlow Version:", tf.__version__)

In [ ]:
# GPU Configuration
print("=" * 60)
print("GPU CONFIGURATION")
print("=" * 60)
print(f"TensorFlow version: {tf.__version__}")
print(f"Num GPUs Available: {len(tf.config.list_physical_devices('GPU'))}")

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"✓ GPU detected and configured")
    except RuntimeError as e:
        print(f"GPU configuration error: {e}")
else:
    print("⚠ No GPU detected - Running on CPU")
print("=" * 60 + "\n")

## Configuration

In [ ]:
# Federated Learning Configuration
NUM_CLIENTS = 100
NUM_ROUNDS = 100  # Total rounds with progressive removal
LOCAL_EPOCHS = 10
BATCH_SIZE = 32
NUM_CLASSES = 10

# Progressive Removal Parameters
REMOVAL_INTERVAL = 10  # Analyze and remove samples every N rounds
REMOVAL_PER_CYCLE = 0.025  # Remove worst 2.5% per cycle
NUM_REMOVAL_CYCLES = 5  # Total of 5 removal cycles (rounds 10, 20, 30, 40, 50)
ANALYSIS_WINDOW = 10  # Analyze last 10 rounds of data
MIN_SAMPLES_PER_CLASS = 6  # Never go below 6 samples per class

# Detection Thresholds (more aggressive for quick detection)
HIGH_LOSS_PERCENTILE = 0.85  # Top 15% worst losses
LOW_CONFIDENCE_THRESHOLD = 0.45  # Confidence below 45%
HIGH_VARIANCE_THRESHOLD = 0.25  # Prediction variance threshold

# Directories
DATA_DIR = 'mnist_100_clients'
RESULTS_DIR = 'results_hard_sample_removal'
os.makedirs(RESULTS_DIR, exist_ok=True)
print("FEDERATED LEARNING - PROGRESSIVE HARD SAMPLE REMOVAL")
print("=" * 60)
print("FEDERATED LEARNING - HARD SAMPLE REMOVAL")
print("=" * 60)
print(f"Local Epochs per Round: {LOCAL_EPOCHS}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"\nProgressive Removal Strategy:")
print(f"  Removal Interval: Every {REMOVAL_INTERVAL} rounds")
print(f"  Removal per Cycle: {REMOVAL_PER_CYCLE*100:.1f}%")
print(f"  Number of Cycles: {NUM_REMOVAL_CYCLES}")
print(f"  Total Removal: ~{REMOVAL_PER_CYCLE*NUM_REMOVAL_CYCLES*100:.1f}%")
print(f"  Removal Rounds: {[i*REMOVAL_INTERVAL for i in range(1, NUM_REMOVAL_CYCLES+1)]}")
print(f"  Analysis Window: Last {ANALYSIS_WINDOW} rounds")
print(f"  Min Samples per Class: {MIN_SAMPLES_PER_CLASS}")
print(f"\nDetection Criteria:")
print(f"  High Loss: Top {(1-HIGH_LOSS_PERCENTILE)*100:.0f}% worst")
print(f"  Low Confidence: < {LOW_CONFIDENCE_THRESHOLD*100:.0f}%")

print(f"  High Variance: > {HIGH_VARIANCE_THRESHOLD}")
print(f"Results Directory: {RESULTS_DIR}/")print("=" * 60 + "\n")

print(f"\nData Directory: {DATA_DIR}/")
print("=" * 60 + "\n")print(f"Results Directory: {RESULTS_DIR}/")

## Load Data

In [ ]:
# Load test data (common for all clients)
print("Loading common test dataset...")
test_file = os.path.join(DATA_DIR, 'test_500_samples.npz')
test_data = np.load(test_file)

x_test = test_data['x'] / 255.0
y_test = test_data['y']
x_test = x_test.reshape(len(x_test), 28*28)

print(f"✓ Test data loaded: {x_test.shape}")
print(f"  Labels shape: {y_test.shape}")

In [ ]:
# Load all client data
print(f"\nLoading data for {NUM_CLIENTS} clients...")
client_data = []

for client_id in range(1, NUM_CLIENTS + 1):
    client_file = os.path.join(DATA_DIR, f'client_{client_id}.npz')
    data = np.load(client_file)
    
    x_client = data['x'] / 255.0
    y_client = data['y']
    x_client = x_client.reshape(len(x_client), 28*28)
    
    client_data.append({
        'x_train': x_client,
        'y_train': y_client,
        'x_test': x_test,
        'y_test': y_test,
        'x_train_original': x_client.copy(),  # Keep original for comparison
        'y_train_original': y_client.copy()
    })
    
    if client_id % 20 == 0:
        print(f"  Loaded {client_id}/{NUM_CLIENTS} clients")

print(f"\n✓ All {NUM_CLIENTS} clients loaded successfully")
print(f"  Each client has {len(client_data[0]['x_train'])} training samples")
print(f"  Common test set: {len(x_test)} samples")

## Model Architecture

In [ ]:
# Define model
def create_model():
    """Lightweight model optimized for small datasets"""
    model = keras.Sequential([
        keras.layers.Dense(64, input_shape=(784,), activation="relu"),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(32, activation="relu"),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(10, activation="softmax")
    ])
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

# Test model creation
print("Testing model architecture...")
test_model = create_model()
test_model.summary()
print("\n✓ Model architecture validated")

## Hard Sample Detection Functions

In [ ]:
# Functions for detecting hard samples
def compute_sample_loss(model, x_samples, y_samples):
    """
    Compute loss for each individual sample
    Returns: array of losses, one per sample
    """
    predictions = model.predict(x_samples, verbose=0)
    losses = []
    
    for i in range(len(x_samples)):
        # Compute categorical crossentropy for this sample
        true_class = int(y_samples[i])
        pred_prob = predictions[i][true_class]
        loss = -np.log(pred_prob + 1e-7)  # Add small epsilon to avoid log(0)
        losses.append(loss)
    
    return np.array(losses)

def compute_sample_confidence(model, x_samples):
    """
    Compute prediction confidence for each sample
    Returns: array of confidences (max probability), one per sample
    """
    predictions = model.predict(x_samples, verbose=0)
    confidences = np.max(predictions, axis=1)
    return confidences

def compute_prediction_variance(predictions_history):
    """
    Compute how much predictions vary across rounds for each sample
    predictions_history: list of prediction arrays (one per round)
    Returns: variance score for each sample
    """
    # Stack predictions: shape (rounds, samples, classes)
    stacked = np.array(predictions_history)
    
    # Get predicted class for each round
    pred_classes = np.argmax(stacked, axis=2)  # shape: (rounds, samples)
    
    # Compute variance of predicted classes
    variances = []
    for sample_idx in range(pred_classes.shape[1]):
        sample_preds = pred_classes[:, sample_idx]
        # Use standard deviation of predictions as inconsistency measure
        variance = np.std(sample_preds) / NUM_CLASSES  # Normalize by number of classes
        variances.append(variance)
    
    return np.array(variances)

def identify_hard_samples(losses_history, confidences_history, predictions_history, 
                          y_train, removal_percentage, min_per_class):
    """
    Identify hard samples based on multiple criteria from recent rounds
    Returns: boolean mask (True = keep sample, False = remove sample)
    """
    num_samples = len(y_train)
    
    if len(losses_history) == 0:
        # No data yet, keep all samples
        return np.ones(num_samples, dtype=bool), np.zeros(num_samples)
    
    # Compute statistics across recent rounds
    avg_loss = np.mean(losses_history, axis=0)
    avg_confidence = np.mean(confidences_history, axis=0)
    pred_variance = compute_prediction_variance(predictions_history) if len(predictions_history) > 1 else np.zeros(num_samples)
    
    # Normalize scores to [0, 1] with more aggressive thresholds
    loss_score = (avg_loss - avg_loss.min()) / (avg_loss.max() - avg_loss.min() + 1e-7)
    conf_score = 1.0 - avg_confidence  # Lower confidence = higher score
    var_score = pred_variance / (pred_variance.max() + 1e-7) if pred_variance.max() > 0 else np.zeros(num_samples)
    
    # Combined hardness score (weighted average)
    hardness_score = 0.4 * loss_score + 0.4 * conf_score + 0.2 * var_score
    
    # Determine how many samples to remove
    num_to_remove = int(num_samples * removal_percentage)
    num_to_keep = num_samples - num_to_remove
    
    # Get indices sorted by hardness (highest = hardest)
    sorted_indices = np.argsort(hardness_score)[::-1]
    
    # Initialize mask (True = keep)
    keep_mask = np.ones(num_samples, dtype=bool)
    
    # Try to remove hardest samples, but respect per-class minimums
    samples_per_class = defaultdict(list)
    for idx, label in enumerate(y_train):
        samples_per_class[int(label)].append(idx)
    
    removed_count = 0
    for idx in sorted_indices:
        if removed_count >= num_to_remove:
            break
        
        # Check if removing this sample would violate per-class minimum
        label = int(y_train[idx])
        current_count = sum([keep_mask[i] for i in samples_per_class[label]])
        
        if current_count > min_per_class:
            keep_mask[idx] = False
            removed_count += 1

    print("✓ Hard sample detection functions defined")

    return keep_mask, hardness_score

## Federated Learning Functions

In [ ]:
# Federated Averaging function
def federated_averaging(weights_list):
    """Average weights from all clients (FedAvg)"""
    avg_weights = []
    for weights_tuple in zip(*weights_list):
        avg_weights.append(np.mean(weights_tuple, axis=0))
    return avg_weights

print("✓ Federated averaging function defined")

## Initialize Federated Learning

In [ ]:
# Initialize global model
print("\n" + "=" * 60)
print("INITIALIZING FEDERATED LEARNING")
print("=" * 60)

global_model = create_model()
global_weights = global_model.get_weights()

# Tracking arrays
client_train_acc_history = [[] for _ in range(NUM_CLIENTS)]
client_test_acc_history = [[] for _ in range(NUM_CLIENTS)]
client_best_weights = [None for _ in range(NUM_CLIENTS)]
client_best_test_acc = [0.0 for _ in range(NUM_CLIENTS)]
client_best_train_acc = [0.0 for _ in range(NUM_CLIENTS)]
client_rejections = [[] for _ in range(NUM_CLIENTS)]

# Hard sample tracking (rolling window for progressive removal)
client_sample_losses = [[] for _ in range(NUM_CLIENTS)]  # Rolling window of loss arrays
client_sample_confidences = [[] for _ in range(NUM_CLIENTS)]  # Rolling window of confidence arrays
client_sample_predictions = [[] for _ in range(NUM_CLIENTS)]  # Rolling window of prediction arrays
client_removed_samples = [0 for _ in range(NUM_CLIENTS)]  # Cumulative count of removed samples
client_samples_kept = [None for _ in range(NUM_CLIENTS)]  # Current mask of kept samples
client_removal_history = [[] for _ in range(NUM_CLIENTS)]  # Track removals per cycle
removal_rounds = []  # Track which rounds had removals

print("✓ Progressive hard sample removal enabled")
print(f"✓ Removal scheduled for rounds: {[i*REMOVAL_INTERVAL for i in range(1, NUM_REMOVAL_CYCLES+1)]}")

print("✓ Hard sample detection enabled")print("✓ Sample statistics collection ready")

## Phase 1: Profiling Phase (Collect Sample Statistics)

In [ ]:
# Profiling Phase: Collect statistics about each sample
print("\n" + "=" * 60)
print("PHASE 1: PROFILING - IDENTIFYING HARD SAMPLES")
print("=" * 60 + "\n")

for round_num in tqdm(range(PROFILING_ROUNDS), desc="Profiling Rounds", unit="round"):
    print(f"\n{'='*60}")
    print(f"PROFILING ROUND {round_num + 1}/{PROFILING_ROUNDS}")
    print(f"{'='*60}")
    
    local_weights = []
    round_train_accs = [0.0] * NUM_CLIENTS
    round_test_accs = [0.0] * NUM_CLIENTS
    round_rejections = 0
    round_acceptances = 0
    
    # Check if this is a removal round
    is_removal_round = (round_num + 1) % REMOVAL_INTERVAL == 0 and (round_num + 1) <= NUM_REMOVAL_CYCLES * REMOVAL_INTERVAL
    
    # Train all clients
    for client_id in tqdm(range(NUM_CLIENTS), desc=f"Training Clients (Round {round_num + 1})", 
                         unit="client", leave=False):
        
        client_model = create_model()
        client_model.set_weights(global_weights)
        
        x_train = client_data[client_id]['x_train']
        y_train = client_data[client_id]['y_train']
        # Train with current data
        y_test = client_data[client_id]['y_test']
        
        # Train normally
        history = client_model.fit(
            x_train, y_train,
            epochs=LOCAL_EPOCHS,
            batch_size=BATCH_SIZE,
        # Collect sample-level statistics for analysis
        )
        
        # Collect sample-level statistics
        sample_losses = compute_sample_loss(client_model, x_train, y_train)
        # Maintain rolling window (keep last ANALYSIS_WINDOW rounds)
        client_sample_losses[client_id].append(sample_losses)
        client_sample_confidences[client_id].append(sample_confidences)
        client_sample_predictions[client_id].append(sample_predictions)
        
        if len(client_sample_losses[client_id]) > ANALYSIS_WINDOW:
            client_sample_losses[client_id] = client_sample_losses[client_id][-ANALYSIS_WINDOW:]
            client_sample_confidences[client_id] = client_sample_confidences[client_id][-ANALYSIS_WINDOW:]
            client_sample_predictions[client_id] = client_sample_predictions[client_id][-ANALYSIS_WINDOW:]
        
        # Evaluate
        train_acc = history.history['accuracy'][-1]
        _, test_acc = client_model.evaluate(x_test, y_test, verbose=0)
        
        # Weight acceptance/rejection logic
        trained_weights = client_model.get_weights()
        
        if test_acc > client_best_test_acc[client_id]:
            client_best_weights[client_id] = [w.copy() for w in trained_weights]
            client_best_test_acc[client_id] = test_acc
            client_best_train_acc[client_id] = train_acc
            client_rejections[client_id].append(0)
            round_acceptances += 1
        else:
            if client_best_weights[client_id] is None:
                client_best_weights[client_id] = [w.copy() for w in trained_weights]
                client_best_test_acc[client_id] = test_acc
                client_best_train_acc[client_id] = train_acc
                client_rejections[client_id].append(0)
                round_acceptances += 1
            else:
                client_rejections[client_id].append(1)
                round_rejections += 1
                test_acc = client_best_test_acc[client_id]
                train_acc = client_best_train_acc[client_id]
        
        # Store for averaging
        local_weights.append(client_best_weights[client_id])
        
        # Store accuracies
        client_train_acc_history[client_id].append(train_acc)
        client_test_acc_history[client_id].append(test_acc)
        round_train_accs[client_id] = train_acc
    # Progressive sample removal at specified intervals
    if is_removal_round:
        print(f"\n🗑️  REMOVAL CYCLE {(round_num + 1) // REMOVAL_INTERVAL}/{NUM_REMOVAL_CYCLES}")
        print("=" * 60)
        
        total_removed_this_cycle = 0
        
        for client_id in range(NUM_CLIENTS):
            x_train = client_data[client_id]['x_train']
            y_train = client_data[client_id]['y_train']
            
            samples_before = len(y_train)
            
            # Identify hard samples based on recent performance
            keep_mask, hardness_scores = identify_hard_samples(

                client_sample_losses[client_id],
    print(f"   Test Accuracy Range: [{min_test_acc:.2f}%, {max_test_acc:.2f}%]")
print("\n" + "="*60)print("="*60 + "\n")

                client_sample_confidences[client_id],
    print(f"   Weights Accepted: {round_acceptances}, Rejected: {round_rejections}")
print("PROFILING PHASE COMPLETE!")print("PROGRESSIVE TRAINING COMPLETE!")

                client_sample_predictions[client_id],

print("="*60 + "\n")print("\n" + "="*60)

                y_train,
print("\n" + "="*60)

                REMOVAL_PER_CYCLE,
print("PROFILING PHASE COMPLETE!")        print(f"   🗑️  Samples Removed This Round: {total_removed_this_cycle}")

                MIN_SAMPLES_PER_CLASS
print("="*60 + "\n")    if is_removal_round:

            )    print(f"   Weights Accepted: {round_acceptances}, Rejected: {round_rejections}")

                print(f"   Test Accuracy Range: [{min_test_acc:.2f}%, {max_test_acc:.2f}%]")

            # Apply mask to remove hard samples    print(f"   Avg Test Accuracy: {avg_test_acc:.2f}%")

            x_train_cleaned = x_train[keep_mask]    print(f"   Avg Training Accuracy: {avg_train_acc:.2f}%")

            y_train_cleaned = y_train[keep_mask]    print(f"\n📊 Round {round_num + 1} Summary:")

                

            # Update client data    max_test_acc = np.max(round_test_accs) * 100

            client_data[client_id]['x_train'] = x_train_cleaned    min_test_acc = np.min(round_test_accs) * 100

            client_data[client_id]['y_train'] = y_train_cleaned    avg_test_acc = np.mean(round_test_accs) * 100

                avg_train_acc = np.mean(round_train_accs) * 100

            # Reset statistics for new dataset    # Round summary

            client_sample_losses[client_id] = []    

            client_sample_confidences[client_id] = []        print("=" * 60)

            client_sample_predictions[client_id] = []        print(f"   Cumulative total: {sum(client_removed_samples)} samples")

                    print(f"   Avg per client: {total_removed_this_cycle/NUM_CLIENTS:.1f}")

            samples_after = len(y_train_cleaned)        print(f"✓ Removed {total_removed_this_cycle} samples across all clients")

            removed_this_cycle = samples_before - samples_after        removal_rounds.append(round_num + 1)

            client_removed_samples[client_id] += removed_this_cycle        

            client_removal_history[client_id].append(removed_this_cycle)            total_removed_this_cycle += removed_this_cycle

## Analyze and Remove Hard Samples

In [ ]:
# Analyze collected statistics and remove hard samples
print("\n" + "=" * 60)
print("ANALYZING SAMPLES AND REMOVING HARD SAMPLES")
print("=" * 60 + "\n")

total_samples_before = 0
total_samples_after = 0
removal_stats = []

for client_id in range(NUM_CLIENTS):
    x_train = client_data[client_id]['x_train']
    y_train = client_data[client_id]['y_train']
    
    samples_before = len(y_train)
    total_samples_before += samples_before
    
    # Identify hard samples
    keep_mask, hardness_scores = identify_hard_samples(
        client_sample_losses[client_id],
        client_sample_confidences[client_id],
        client_sample_predictions[client_id],
        y_train,
        REMOVAL_PERCENTAGE,
        MIN_SAMPLES_PER_CLASS
    )
    
    # Apply mask to remove hard samples
    x_train_cleaned = x_train[keep_mask]
    y_train_cleaned = y_train[keep_mask]
    
    # Update client data
    client_data[client_id]['x_train'] = x_train_cleaned
    client_data[client_id]['y_train'] = y_train_cleaned
    client_samples_kept[client_id] = keep_mask
    
    samples_after = len(y_train_cleaned)
    removed = samples_before - samples_after
    client_removed_samples[client_id] = removed
    total_samples_after += samples_after
    
    removal_stats.append({
        'client_id': client_id + 1,
        'before': samples_before,
        'after': samples_after,
        'removed': removed,
        'percentage': (removed / samples_before) * 100
    })
    
    if (client_id + 1) % 20 == 0:
        print(f"  Processed {client_id + 1}/{NUM_CLIENTS} clients")

print(f"\n✓ Hard sample removal complete!")
print(f"\n📊 Removal Statistics:")
print(f"   Total samples before: {total_samples_before}")
print(f"   Total samples after: {total_samples_after}")
print(f"   Total removed: {total_samples_before - total_samples_after}")
print(f"   Average removal per client: {np.mean([s['removed'] for s in removal_stats]):.1f} samples")
print(f"   Removal percentage: {((total_samples_before - total_samples_after) / total_samples_before)*100:.2f}%")
print(f"   Min samples per client: {min([s['after'] for s in removal_stats])}")
print(f"   Max samples per client: {max([s['after'] for s in removal_stats])}")

# Show per-class statistics for a sample client
sample_client_id = 0
y_original = client_data[sample_client_id]['y_train_original']
y_cleaned = client_data[sample_client_id]['y_train']
print(f"\n📋 Sample Client 1 - Per-Class Breakdown:")
for class_idx in range(NUM_CLASSES):
    before = np.sum(y_original == class_idx)
    after = np.sum(y_cleaned == class_idx)
    print(f"   Class {class_idx}: {before} → {after} ({before-after} removed)")

print("=" * 60 + "\n")

## Phase 2: Training with Cleaned Data

In [ ]:
# Training Phase: Continue with cleaned data
print("\n" + "=" * 60)
print("PHASE 2: TRAINING WITH CLEANED DATA")
print("=" * 60 + "\n")

training_rounds = NUM_ROUNDS - PROFILING_ROUNDS

for round_num in tqdm(range(training_rounds), desc="Training Rounds", unit="round"):
    actual_round = PROFILING_ROUNDS + round_num
    print(f"\n{'='*60}")
    print(f"TRAINING ROUND {actual_round + 1}/{NUM_ROUNDS} (Cleaned Data)")
    print(f"{'='*60}")
    
    local_weights = []
    round_train_accs = [0.0] * NUM_CLIENTS
    round_test_accs = [0.0] * NUM_CLIENTS
    round_rejections = 0
    round_acceptances = 0
    
    # Train all clients with cleaned data
    for client_id in tqdm(range(NUM_CLIENTS), desc=f"Training Clients (Round {actual_round + 1})", 
                         unit="client", leave=False):
        
        client_model = create_model()
        client_model.set_weights(global_weights)
        
        # Use cleaned data
        x_train = client_data[client_id]['x_train']
        y_train = client_data[client_id]['y_train']
        x_test = client_data[client_id]['x_test']
        y_test = client_data[client_id]['y_test']
        
        # Train with cleaned data
        history = client_model.fit(
            x_train, y_train,
            epochs=LOCAL_EPOCHS,
            batch_size=BATCH_SIZE,
            verbose=0
        )
        
        # Evaluate
        train_acc = history.history['accuracy'][-1]
        _, test_acc = client_model.evaluate(x_test, y_test, verbose=0)
        
        # Weight acceptance/rejection logic
        trained_weights = client_model.get_weights()
        
        if test_acc > client_best_test_acc[client_id]:
            client_best_weights[client_id] = [w.copy() for w in trained_weights]
            client_best_test_acc[client_id] = test_acc
            client_best_train_acc[client_id] = train_acc
            client_rejections[client_id].append(0)
            round_acceptances += 1
        else:
            client_rejections[client_id].append(1)
            round_rejections += 1
            test_acc = client_best_test_acc[client_id]
            train_acc = client_best_train_acc[client_id]
        
        # Store for averaging
        local_weights.append(client_best_weights[client_id])
        
        # Store accuracies
        client_train_acc_history[client_id].append(train_acc)
        client_test_acc_history[client_id].append(test_acc)
        round_train_accs[client_id] = train_acc
        round_test_accs[client_id] = test_acc
    
    # Federated averaging
    global_weights = federated_averaging(local_weights)
    global_model.set_weights(global_weights)
    
    # Round summary
    avg_train_acc = np.mean(round_train_accs) * 100
    avg_test_acc = np.mean(round_test_accs) * 100
    min_test_acc = np.min(round_test_accs) * 100
    max_test_acc = np.max(round_test_accs) * 100
    
    print(f"\n📊 Round {actual_round + 1} Summary:")
    print(f"   Avg Training Accuracy: {avg_train_acc:.2f}%")
    print(f"   Avg Test Accuracy: {avg_test_acc:.2f}%")
    print(f"   Test Accuracy Range: [{min_test_acc:.2f}%, {max_test_acc:.2f}%]")
    print(f"   Weights Accepted: {round_acceptances}, Rejected: {round_rejections}")

print("\n" + "="*60)
print("TRAINING WITH CLEANED DATA COMPLETE!")
print("="*60 + "\n")

## Results Analysis

In [ ]:
# Calculate final accuracies and statistics
final_train_accs = [client_train_acc_history[i][-1] * 100 for i in range(NUM_CLIENTS)]
final_test_accs = [client_test_acc_history[i][-1] * 100 for i in range(NUM_CLIENTS)]

avg_final_train = np.mean(final_train_accs)
avg_final_test = np.mean(final_test_accs)

# Calculate improvement from first removal
if len(removal_rounds) > 0:
    first_removal_round = removal_rounds[0] - 1  # Index before first removal
    before_removal_accs = [client_test_acc_history[i][first_removal_round] * 100 for i in range(NUM_CLIENTS)]
    avg_before_removal = np.mean(before_removal_accs)
else:
    avg_before_removal = final_test_accs[0]

improvement = avg_final_test - avg_before_removal
print("FINAL RESULTS - PROGRESSIVE HARD SAMPLE REMOVAL")
# Rejection statistics
total_rejections_per_client = [sum(client_rejections[i]) for i in range(NUM_CLIENTS)]

print("=" * 60)
print("FINAL RESULTS - HARD SAMPLE REMOVAL")
print("=" * 60)
print(f"Overall Performance:")
print(f"  Average Final Training Accuracy: {avg_final_train:.2f}%")
if len(removal_rounds) > 0:
    print(f"  Avg Accuracy Before First Removal: {avg_before_removal:.2f}%")
    print(f"  Avg Accuracy After Progressive Cleaning: {avg_final_test:.2f}%")
    print(f"  Improvement: {improvement:+.2f}% ({'+' if improvement > 0 else ''}{(improvement/avg_before_removal)*100:.2f}% relative)")
else:
    print(f"  No removals performed")

print(f"\nSample Removal Statistics:")
print(f"  Total Samples Removed: {sum(client_removed_samples)}")
print(f"  Avg Samples Removed per Client: {np.mean(client_removed_samples):.1f}")
print(f"  Overall Removal Rate: {(sum(client_removed_samples)/total_samples_before)*100:.2f}%")

print(f"\nWeight Rejection Statistics:")

print(f"  Total Weight Rejections: {sum(total_rejections_per_client)}")
print("=" * 60)

print(f"  Rejection Rate: {sum(total_rejections_per_client) / (NUM_CLIENTS * NUM_ROUNDS) * 100:.2f}%")
print(f"\nWeight Rejection Statistics:")print(f"  Rejection Rate: {sum(total_rejections_per_client) / (NUM_CLIENTS * NUM_ROUNDS) * 100:.2f}%")

print("=" * 60)print(f"  Total Weight Rejections: {sum(total_rejections_per_client)}")

## Visualizations

In [ ]:
# Plot 1: Test Accuracy for All Clients with Removal Highlighting
print("Creating test accuracy visualization...")

fig, axes = plt.subplots(10, 10, figsize=(30, 30))
fig.suptitle('Test Accuracy - Progressive Hard Sample Removal (100 Clients)', 
             fontsize=20, fontweight='bold', y=0.995)

rounds = range(1, NUM_ROUNDS + 1)

for client_id in range(NUM_CLIENTS):
    row = client_id // 10
    col = client_id % 10
    ax = axes[row, col]
    
    test_accs = [acc * 100 for acc in client_test_acc_history[client_id]]
    final_acc = test_accs[-1]
    improvement = final_acc - test_accs[0]
    removed = client_removed_samples[client_id]
    
    # Color based on number of samples removed
    if removed >= 15:
        color = '#FF6B6B'  # Red for high removal
        title_color = 'red'
    elif removed >= 10:
        color = '#FFA500'  # Orange for medium removal
        title_color = 'orange'
    else:
        color = '#2E86AB'  # Blue for low removal
        title_color = 'blue'
    
    ax.plot(rounds, test_accs, linewidth=2, color=color, alpha=0.8)
    ax.fill_between(rounds, test_accs, alpha=0.3, color=color)
    
    # Mark removal rounds
    for removal_round in removal_rounds:
        ax.axvline(x=removal_round, color='red', linestyle='--', alpha=0.4, linewidth=1)
    
    ax.set_title(f'C{client_id+1} ({improvement:+.1f}%) [-{removed}]', 
                fontsize=8, fontweight='bold', color=title_color)
    
    ax.text(NUM_ROUNDS, final_acc, f'{final_acc:.0f}%', 
            fontsize=7, fontweight='bold', verticalalignment='center',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7))
    
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.set_ylim(0, 100)
    ax.set_xlim(0.5, NUM_ROUNDS + 0.5)
    
    if NUM_ROUNDS <= 100:
        ax.set_xticks([1, 25, 50, 75, 100])
        ax.set_xticklabels(['1', '25', '50', '75', '100'], fontsize=7)
    else:
        ax.set_xticks([1, 50, 100, 150, 200])
        ax.set_xticklabels(['1', '50', '100', '150', '200'], fontsize=7)
    
    ax.set_yticks([0, 25, 50, 75, 100])
    ax.set_yticklabels(['0', '25', '50', '75', '100'], fontsize=7)
    
    if row == 9:
        ax.set_xlabel('Round', fontsize=8)
    if col == 0:
        ax.set_ylabel('Accuracy (%)', fontsize=8)

plot_path = os.path.join(RESULTS_DIR, 'test_accuracy_progressive_removal.png')
plot_path = os.path.join(RESULTS_DIR, 'test_accuracy_hard_sample_removal.png')
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
print(f"✓ Saved: {plot_path}")
plt.show()

In [ ]:
# Plot 2: Average Accuracy Comparison with Phase Marking
print("Creating average accuracy comparison...")

rounds = range(1, NUM_ROUNDS + 1)
avg_test_per_round = [np.mean([client_test_acc_history[i][r] * 100 for i in range(NUM_CLIENTS)]) 
                      for r in range(NUM_ROUNDS)]
min_test = [np.min([client_test_acc_history[i][r] * 100 for i in range(NUM_CLIENTS)]) 
            for r in range(NUM_ROUNDS)]
max_test = [np.max([client_test_acc_history[i][r] * 100 for i in range(NUM_CLIENTS)]) 
            for r in range(NUM_ROUNDS)]

fig, ax = plt.subplots(figsize=(14, 8))

ax.plot(rounds, avg_test_per_round, marker='o', linewidth=2.5, markersize=5, 
        color='#2E86AB', label='Average Test Accuracy', alpha=0.9, markevery=max(1, NUM_ROUNDS//10))
ax.plot(rounds, min_test, linestyle='--', linewidth=1.5, 
        color='gray', label='Min/Max Range', alpha=0.5)
ax.plot(rounds, max_test, linestyle='--', linewidth=1.5, 
        color='gray', alpha=0.5)

ax.fill_between(rounds, min_test, max_test, alpha=0.1, color='gray')

# Mark removal rounds
for idx, removal_round in enumerate(removal_rounds):
    ax.axvline(x=removal_round, color='red', linestyle='--', alpha=0.6, linewidth=2)
    if idx == 0:
        ax.text(removal_round, 5, f'↓ Cycle {idx+1}', fontsize=8, ha='center', 
               color='red', fontweight='bold')
    else:
        ax.text(removal_round, 5 + (idx % 2) * 3, f'↓ {idx+1}', fontsize=8, ha='center', 
               color='red', fontweight='bold')

ax.text(NUM_ROUNDS/2, 95, f'Progressive Removal\n{len(removal_rounds)} cycles, {sum(client_removed_samples)} total samples', 
       fontsize=10, ha='center', bbox=dict(boxstyle='round,pad=0.5', facecolor='yellow', alpha=0.6))

ax.set_xlabel('Communication Round', fontsize=12, fontweight='bold')
ax.set_title('Federated Learning - Progressive Hard Sample Removal\nTest Accuracy Over Time', 
ax.set_title('Federated Learning - Hard Sample Removal Strategy\nTest Accuracy Over Time', 
            fontsize=14, fontweight='bold')
ax.legend(fontsize=10, loc='lower right')
ax.grid(True, alpha=0.3, linestyle='--')
ax.set_ylim(0, 100)

plt.tight_layout()
avg_plot_path = os.path.join(RESULTS_DIR, 'average_accuracy_comparison.png')
plt.savefig(avg_plot_path, dpi=300, bbox_inches='tight')
print(f"✓ Saved: {avg_plot_path}")
plt.show()

In [ ]:
# Plot 3: Sample Removal Distribution
print("Creating sample removal distribution...")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 3a: Histogram of samples removed per client
ax1 = axes[0]
removal_counts = [client_removed_samples[i] for i in range(NUM_CLIENTS)]
ax1.hist(removal_counts, bins=20, color='#FF6B6B', alpha=0.7, edgecolor='black')
ax1.axvline(np.mean(removal_counts), color='blue', linestyle='--', linewidth=2, 
           label=f'Mean: {np.mean(removal_counts):.1f}')
ax1.set_xlabel('Number of Samples Removed', fontsize=11, fontweight='bold')
ax1.set_ylabel('Number of Clients', fontsize=11, fontweight='bold')
ax1.set_title('Distribution of Sample Removal', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Plot 3b: Scatter plot - samples removed vs final accuracy
ax2 = axes[1]
removal_counts = [client_removed_samples[i] for i in range(NUM_CLIENTS)]
final_accs = [client_test_acc_history[i][-1] * 100 for i in range(NUM_CLIENTS)]

scatter = ax2.scatter(removal_counts, final_accs, c=final_accs, cmap='RdYlGn', 
                     s=50, alpha=0.7, edgecolors='black', linewidth=0.5)
ax2.set_xlabel('Number of Samples Removed', fontsize=11, fontweight='bold')
ax2.set_ylabel('Final Test Accuracy (%)', fontsize=11, fontweight='bold')
ax2.set_title('Sample Removal vs Final Accuracy', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)

# Add colorbar
cbar = plt.colorbar(scatter, ax=ax2)
cbar.set_label('Accuracy (%)', fontsize=10)

# Calculate correlation
correlation = np.corrcoef(removal_counts, final_accs)[0, 1]
ax2.text(0.05, 0.95, f'Correlation: {correlation:.3f}', 
        transform=ax2.transAxes, fontsize=10, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
dist_plot_path = os.path.join(RESULTS_DIR, 'sample_removal_distribution.png')
plt.savefig(dist_plot_path, dpi=300, bbox_inches='tight')
print(f"✓ Saved: {dist_plot_path}")
plt.show()

print("\n" + "="*60)
print("ALL VISUALIZATIONS COMPLETE!")
print("="*60)

## Save Results

In [ ]:
# Save results summary
results_file = os.path.join(RESULTS_DIR, 'final_results_hard_sample_removal.txt')

with open(results_file, 'w') as f:
    f.write("=" * 60 + "\n")
    f.write("FEDERATED LEARNING - PROGRESSIVE HARD SAMPLE REMOVAL\n")
    f.write("=" * 60 + "\n\n")
    
    f.write("Configuration:\n")
    f.write(f"  Number of Clients: {NUM_CLIENTS}\n")
    f.write(f"  Total Rounds: {NUM_ROUNDS}\n")
    f.write(f"  Removal Interval: Every {REMOVAL_INTERVAL} rounds\n")
    f.write(f"  Removal per Cycle: {REMOVAL_PER_CYCLE*100:.1f}%\n")
    f.write(f"  Number of Cycles: {NUM_REMOVAL_CYCLES}\n")
    f.write(f"  Removal Rounds: {removal_rounds}\n")
    f.write(f"  Local Epochs: {LOCAL_EPOCHS}\n")
    f.write(f"  Min Samples per Class: {MIN_SAMPLES_PER_CLASS}\n\n")
    
    f.write("Overall Performance:\n")
    f.write(f"  Average Final Test Accuracy: {avg_final_test:.2f}%\n")
    f.write(f"  Test Accuracy Range: [{np.min(final_test_accs):.2f}%, {np.max(final_test_accs):.2f}%]\n")
    f.write(f"  Standard Deviation: {np.std(final_test_accs):.2f}%\n\n")
    
    if len(removal_rounds) > 0:
        f.write(f"  Avg Accuracy Before First Removal: {avg_before_removal:.2f}%\n")
        f.write(f"  Avg Accuracy After Progressive Cleaning: {avg_final_test:.2f}%\n")
        f.write(f"  Absolute Improvement: {improvement:+.2f}%\n")
        f.write(f"  Relative Improvement: {(improvement/avg_before_removal)*100:+.2f}%\n\n")
    else:
    f.write(f"  Original Total Samples: {total_samples_original}\n")
    f.write(f"  Final Total Samples: {total_samples_final}\n")
    f.write(f"  Total Samples Removed: {total_removed}\n")
    f.write(f"  Overall Removal Rate: {(total_removed/total_samples_original)*100:.2f}%\n")
    f.write(f"  Final Total Samples: {total_samples_after}\n")
    f.write(f"  Total Samples Removed: {sum(client_removed_samples)}\n")
    f.write(f"  Overall Removal Rate: {(sum(client_removed_samples)/total_samples_before)*100:.2f}%\n")
    f.write(f"  Avg Removed per Client: {np.mean(client_removed_samples):.2f}\n")
    f.write(f"  Min Removed: {min(client_removed_samples)}\n")
    f.write(f"  Max Removed: {max(client_removed_samples)}\n\n")
    
    f.write("Top 10 Clients by Final Accuracy:\n")
    top_clients = sorted(enumerate(final_test_accs), key=lambda x: x[1], reverse=True)[:10]
    for rank, (client_id, acc) in enumerate(top_clients, 1):
        removed = client_removed_samples[client_id]
        f.write(f"  {rank}. Client {client_id+1}: {acc:.2f}% (removed {removed} samples)\n")
    
    f.write("\nBottom 10 Clients by Final Accuracy:\n")
    bottom_clients = sorted(enumerate(final_test_accs), key=lambda x: x[1])[:10]
    for rank, (client_id, acc) in enumerate(bottom_clients, 1):
        removed = client_removed_samples[client_id]
        f.write(f"  {rank}. Client {client_id+1}: {acc:.2f}% (removed {removed} samples)\n")
    
    f.write("\n" + "=" * 60 + "\n")

print(f"✓ Results saved to: {results_file}")

print("\n" + "="*60)
print("="*60)print("="*60)
print("ALL TASKS COMPLETE!")